In [2]:
import pandas as pd
import requests
import time
from __future__ import annotations

#sectors requester
def sectors_requester(endpoint, params=None):
  headers = {"Authorization": "8f78cc2a4fa85eb0606e28cf870f93d855d5f8a9b71afa51c2eec738c5147369"}
  url = "https://api.sectors.app/v2/"

  base_url = url + endpoint + "/"
  response = requests.get(base_url, headers=headers, params=params)
  response.raise_for_status()
  data = response.json()

  return data

#corporate actions sections
def getting_companies_corporate_actions(ticker_list):
    response_list = []
    for symbol in ticker_list:
        endpoint = 'company/corporate-actions'
        endpoint = endpoint + f'/{symbol}'
        response = sectors_requester(endpoint)
        response_list.append(response)
        time.sleep(10)

    return response_list

#scrapping corporate actions
def scraping_corporate_actions_data(response_list):
    event_types = list(response_list[0]['corporate_actions'])
    tables = {}
    for et in event_types:
        rows = []
        for entry in response_list:
            symbol = entry["symbol"]
            records = entry["corporate_actions"].get(et)
            if not records:
                continue
            for r in records:
                rows.append({"symbol": symbol, **r})
        tables[et] = pd.DataFrame(rows)

    long_df = pd.concat([df.assign(event_type=et) for et, df in tables.items() if not df.empty],axis=0, ignore_index=True)
    # 3. put the classifier columns up front for readability
    cols = ["symbol", "event_type"] + [c for c in long_df.columns if c not in ("symbol", "event_type")]
    long_df = long_df[cols]
    return long_df

#revenue segments sections, take company has revenue segments 
def companies_with_revenue_segments():
    endpoint = 'companies/list_companies_with_segments/'
    response = sectors_requester(endpoint)
    return response

#getting the targeted company revenue segments
def get_companies_revenue_segments(targeted_companies_ticker_list):
    companies_with_revenue_segments_response = companies_with_revenue_segments()
    list_of_response = []
    list_of_companies = list(companies_with_revenue_segments_response)

    for ticker in targeted_companies_ticker_list:
        if ticker in list_of_companies:
            endpoint = f'company/get-segments/{ticker}'
            list_of_financial_year = companies_with_revenue_segments_response[ticker]['financial_year']
            for financial_year in list_of_financial_year:
                params = {'financial_year' : financial_year}
                response = sectors_requester(endpoint, params)
                list_of_response.append(response)
                time.sleep(10)
    return list_of_response

#scrapping the responses targeted revenue segments
def scrapping_company_revenue_segment(company_revenue_segment_response):
    list_of_dataframe = []
    for industri in company_revenue_segment_response:
        ticker = industri['symbol']
        revenue_breakdown = pd.DataFrame(columns=['company_name','value', 'source', 'target'])
        company_name = []
        value = []
        source = []
        target = []
        for i in industri['revenue_breakdown'] : 
            company_name.append(ticker)
            value.append(i['value'])
            source.append(i['source'])
            target.append(i['target'])
        revenue_breakdown['company_name'] = company_name
        revenue_breakdown['value'] = value
        revenue_breakdown['source'] = source
        revenue_breakdown['target'] = target
        revenue_breakdown['financial_year'] = industri['financial_year']

        list_of_dataframe.append(revenue_breakdown)

    final_df = pd.concat(list_of_dataframe,axis=0)
    return final_df

#getting the shareholder composition
def get_shareholder_composition(ticker_list):
    response_list = []
    for symbol in ticker_list:
        endpoint = 'company/shareholders-composition'
        endpoint = endpoint + f'/{symbol}'
        current_year = time.localtime().tm_year
        for financial_year in [current_year-1, current_year]:
            params = {'year' : financial_year}
            response = sectors_requester(endpoint, params)
            response_list.append(response)
            time.sleep(10)

    return response_list

#scrapping the company shareholder composition
def scraping_shareholder_composition(response_list):
    list_of_dataframe = []
    previous_ticker = ''
    for i in range(len(response_list)):
        symbol = response_list[i]['symbol']
        dataframe = pd.DataFrame(response_list[i]['data'])
        dataframe['symbol'] = symbol
        if symbol == previous_ticker:
            dataframe = pd.concat([list_of_dataframe[len(list_of_dataframe)-1], dataframe], axis=0, ignore_index=True)
            list_of_dataframe[len(list_of_dataframe)-1] = dataframe
        else:
            previous_ticker = symbol
            previous_dataframe = dataframe
            list_of_dataframe.append(dataframe)
    df_final = pd.concat(list_of_dataframe, axis=0, ignore_index=True)
    return df_final

#pulling company report data out of the api
import time
def company_report(ticker_list):
    response_list = []
    for symbol in ticker_list:
        endpoint = 'company/report'
        endpoint = endpoint + f'/{symbol}'
        params = {
            'sections': 'dividend,financials,future,management,overview,ownership,peers,valuation'
        }
        current_year = time.localtime().tm_year
        response = sectors_requester(endpoint, params)
        response_list.append(response)
        time.sleep(10)

    return response_list

#scraping the corporate report data
def _get(d: dict, *path, default=None):
    cur = d
    for key in path:
        if not isinstance(cur, dict) or key not in cur:
            return default
        cur = cur[key]
    return cur if cur is not None else default


def company_report_dataframe(data_list: list[dict]) -> dict[str, pd.DataFrame]:
    overview_rows = []
    valuation_hist_rows = []
    financials_hist_rows = []
    ratio_rows = []
    dividend_rows = []
    dividend_breakdown_rows = []
    forecast_rows = []
    exec_rows = []
    exec_shareholding_rows = []
    shareholder_rows = []
    peer_rows = []

    for entry in data_list:
        symbol = entry.get("symbol")
        company_name = entry.get("company_name")
        key = {"symbol": symbol, "company_name": company_name}

        # ---------- overview (1 row per stock) ----------
        ov = entry.get("overview", {}) or {}
        atp = ov.get("all_time_price", {}) or {}

        def _pt_val(band):
            band = atp.get(band, {}) or {}
            if not band:
                return None, None
            date, val = next(iter(band.items()))
            return date, val

        row = {
            **key,
            "sector": ov.get("sector"),
            "industry": ov.get("industry"),
            "sub_sector": ov.get("sub_sector"),
            "listing_board": ov.get("listing_board"),
            "listing_date": ov.get("listing_date"),
            "market_cap": ov.get("market_cap"),
            "market_cap_rank": ov.get("market_cap_rank"),
            "employee_num": ov.get("employee_num"),
            "esg_score": ov.get("esg_score"),
            "last_close_price": ov.get("last_close_price"),
            "latest_close_date": ov.get("latest_close_date"),
            "daily_close_change": ov.get("daily_close_change"),
            "website": ov.get("website"),
            "tags": ", ".join(ov.get("tags") or []),
            "indices": ", ".join(ov.get("indices") or []),
            "affiliates": ", ".join(ov.get("affiliates") or []),
            "intrinsic_value": _get(entry, "valuation", "intrinsic_value"),
            "forward_pe": _get(entry, "valuation", "forward_pe"),
        }
        for band in ["ytd_low", "ytd_high", "52_w_low", "52_w_high",
                     "90_d_low", "90_d_high", "all_time_low", "all_time_high"]:
            date, val = _pt_val(band)
            row[f"{band}_date"] = date
            row[f"{band}_price"] = val
        overview_rows.append(row)

        # ---------- valuation history (1 row per stock per year) ----------
        for v in _get(entry, "valuation", "historical_valuation", default=[]):
            valuation_hist_rows.append({**key, **v})

        # ---------- financials history (1 row per stock per year) ----------
        for fin in _get(entry, "financials", "historical_financials", default=[]):
            financials_hist_rows.append({**key, **fin})

        # ---------- financial ratios (nested dict -> flat row) ----------
        for r in _get(entry, "financials", "historical_financial_ratio", default=[]):
            flat = {**key, "year": r.get("year")}
            for group, metrics in r.items():
                if group == "year" or not isinstance(metrics, dict):
                    continue
                for metric_name, metric_val in metrics.items():
                    flat[f"{group}_{metric_name}"] = metric_val
            ratio_rows.append(flat)

        # ---------- dividends (1 summary row per stock) ----------
        div = entry.get("dividend", {}) or {}
        dividend_rows.append({
            **key,
            "yield_ttm": div.get("yield_ttm"),
            "dividend_ttm": div.get("dividend_ttm"),
            "payout_ratio": div.get("payout_ratio"),
            "cash_payout_ratio": div.get("cash_payout_ratio"),
            "last_ex_dividend_date": div.get("last_ex_dividend_date"),
            "avg_yield_5y": _get(div, "dividend_yield_avg", "avg_yield"),
        })
        for year, ydata in (div.get("historical_dividends") or {}).items():
            for pay in ydata.get("breakdown", []):
                dividend_breakdown_rows.append({
                    **key, "year": year,
                    "date": pay.get("date"),
                    "amount": pay.get("total"),
                    "yield": pay.get("yield"),
                })

        # ---------- forecasts / analyst ratings (1 row per stock) ----------
        val_fc = (_get(entry, "future", "company_value_forecasts", default=[]) or [{}])[0]
        growth_fc = (_get(entry, "future", "company_growth_forecasts", default=[]) or [{}])[0]
        ratings = _get(entry, "future", "analyst_rating_breakdown", default={}) or {}
        forecast_rows.append({
            **key,
            **{f"forecast_{k}": v for k, v in val_fc.items()},
            **{f"growth_{k}": v for k, v in growth_fc.items()},
            **{f"rating_{k}": v for k, v in ratings.items()},
        })

        # ---------- management ----------
        for e in _get(entry, "management", "key_executives", default=[]):
            exec_rows.append({**key, **e})
        for e in _get(entry, "management", "executives_shareholdings", default=[]):
            exec_shareholding_rows.append({**key, **e})

        # ---------- ownership ----------
        for s in _get(entry, "ownership", "major_shareholders", default=[]):
            shareholder_rows.append({**key, **s})

        # ---------- peers ----------
        for peer_block in entry.get("peers", []) or []:
            companies = _get(peer_block, "peers_data", "companies", default=[])
            group_name = _get(peer_block, "peers_data", "group_name", default={}) or {}
            for c in companies:
                c = dict(c)
                c.pop("point_summaries", None)
                c.pop("int_income_breakdown", None)
                c.pop("operating_expense_breakdown", None)
                peer_rows.append({**key, **group_name, **c})

    return {
        "overview": pd.DataFrame(overview_rows),
        "valuation_history": pd.DataFrame(valuation_hist_rows),
        "financials_history": pd.DataFrame(financials_hist_rows),
        "financial_ratios": pd.DataFrame(ratio_rows),
        "dividends": pd.DataFrame(dividend_rows),
        "dividend_breakdown": pd.DataFrame(dividend_breakdown_rows),
        "forecasts": pd.DataFrame(forecast_rows),
        "executives": pd.DataFrame(exec_rows),
        "executive_shareholdings": pd.DataFrame(exec_shareholding_rows),
        "major_shareholders": pd.DataFrame(shareholder_rows),
        "peers": pd.DataFrame(peer_rows),
    }


def json_dataframe_to_dataframe(filename):
    with open(filename) as f:
        dataframe_json = json.load(f)

    dataframe_json = {key: pd.DataFrame(records) for key, records in dataframe_json.items()}
    return dataframe_json

def saving_json_dataframe(filename, json_dataframe):
    json_dataframe = {key: dataframe.to_dict(orient='records') for key, dataframe in json_dataframe.items()}
    with open(filename, 'w') as f:
        json.dump(json_dataframe, f, indent=2, default=str)

In [5]:
#How to use the class and the method

# fcorporate_actions = '../../dataset/top10-transportation-by-market-cap-corporate-action.csv'
# fcompany_revenue_segments = '../../dataset/csv/Company Revenue Segment for 10 top industires in transportation sector.csv'
# fshareholder_composition = '../../dataset/csv/top10-transportation-by-market-cap-shareholder-composition.csv'
# fcompany_report='../../dataset/json/top10-transportation-by-marketcap-company_report.json'

# #list of companies (just a test)
# df = pd.read_csv('../../dataset/csv/Top 10 transportation companies by market cap.csv')

# #corporate actions secions
# response = getting_companies_corporate_actions(df['symbol'])
# df_final = scraping_corporate_actions_data(response)
# df_final.to_csv(fcorporate_actions, index=False)

# #company revenue segments
# response = get_companies_revenue_segments(df['symbol'])
# final_df = scrapping_company_revenue_segment(response)
# final_df.to_csv(fcompany_revenue_segments,index=False)

# #shareholder composition
# response_list = get_shareholder_composition(df['symbol'])
# df_final = scraping_shareholder_composition(response_list)
# df_final.to_csv(fshareholder_composition,index=False)

# #company report
# complete_response_list = company_report(df['symbol'])
# json_dataframe = company_report_dataframe(complete_response_list)
# saving_json_dataframe(fcompany_report, json_dataframe)